In [1]:
import copy
from copy import deepcopy

import csv
import itertools

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import os


import pybma

In [2]:
#functions to process proof progressions to turn them into a heatmap

def singleton(a):
    if a[0]==a[1]:
        return 1
    else:
        return 0
    
def timeHM(timepoint):
    state = []
    for var in timepoint.keys():
        state.append(singleton(timepoint[var]))
    return state

def timeText(timepoint):
    state = []
    for var in timepoint.keys():
        state.append(str(timepoint[var][0])+"-"+str(timepoint[var][1]))
    return state

In [3]:
def proofProgressionPlot(p,annotate=False):
    # Set font at the start
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    
    progression = p["ProofProgression"]["history"]
    data = np.fliplr(np.array([ timeHM(time[1]) for time in progression ]).transpose())
    # Create custom colormap with your two colors
    colors = ['#E8D4E8', '#C8E8E0']  # purple-pink and mint-green
    cmap = mcolors.ListedColormap(colors)
    
    # Create figure and plot
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # Use pcolormesh instead of imshow - it handles grid lines better
    im = ax.pcolormesh(data, cmap=cmap, vmin=0, vmax=1, edgecolors='white', linewidth=2)
    
    # Add text to each cell
    if annotate:
        text_data = np.fliplr(np.array([ timeText(time[1]) for time in progression ]).transpose())
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                text = ax.text(j + 0.5, i + 0.5, str(text_data[i, j]),
                              ha="center", va="center", color="black", fontsize=10)
    
    # Invert y-axis to match imshow behavior (top to bottom)
    ax.invert_yaxis()
    
    # Remove all ticks and labels
    ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    
    # Remove spines (borders)
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    plt.tight_layout()
    plt.show()
    

In [4]:
def simulation_graph(data_dict, title="Simulation Graph", nameMap=None):
    """
    Create a line graph matching the style shown.
    
    Parameters:
    data_dict: Dictionary with keys as labels and values as lists of y-values
               Example: {'a': [0,0,3,3,2,1,1,1,3,3,3], 'b': [0,0,0,3,3,2,1,1,1,3,3], ...}
    title: Graph title
    """
    # Set font properties
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 11
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot each line
    print(data_dict)
    for label, y_values in data_dict.items():
        x_values = range(len(y_values))
        if nameMap != None:
            label = nameMap[label]
        ax.plot(x_values, y_values, #color=colors.get(label, 'black'), 
                linewidth=2.5, label=label)
    
    # Styling
    ax.set_title(title, fontsize=13, loc='left', pad=15, fontweight='normal')
    ax.grid(True, alpha=0.3, linewidth=0.8)
    ax.set_axisbelow(True)
    
    # Set y-axis to show integer values
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    
    # Add legend with font styling
    ax.legend(loc='best', frameon=False, fontsize=11)
    
    # Style tick labels
    ax.tick_params(labelsize=10)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [5]:
def simulation_table(data_dict, title="Initial Value", nameMap=None ):
    """
    Create a table view of simulation data.
    - Yellow: value changes from previous timestep
    - Gray: state that repeats a previous state (marks all repetitions except first occurrence)
    - White: value same as previous timestep
    
    Parameters:
    data_dict: Dictionary with keys as labels and values as lists of y-values
               Example: {'a': [1,2,3,3,2,1,1,2,3,3], 'b': [0,1,2,3,3,2,1,1,2,3], ...}
    title: Title for the first column
    """
    # Set font properties
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = 11
    
    # Get number of rows and columns
    values = data_dict.values()
    num_rows = len(data_dict)
    max_cols = max(len(row) for row in values) if values else 0
    labels = list(data_dict.keys())
    # Build state for each column
    states = []
    for col_idx in range(max_cols):
        current_state = tuple(data_dict[label][col_idx] for label in labels)
        states.append(current_state)
    
    # Find which state repeats (if any) and mark all its repetitions except first occurrence
    repeated_cols = set()
    state_first_occurrence = {}
    repeating_state = None
    
    for col_idx, state in enumerate(states):
        if state in state_first_occurrence:
            # This state has been seen before, mark this as repeated
            if repeating_state == None or repeating_state == state:
                repeated_cols.add(col_idx)
                repeating_state = state
        else:
            # First time seeing this state
            state_first_occurrence[state] = col_idx
    
    # Create figure - each cell is roughly square
    cell_height = 0.6
    cell_width = 0.6
    header_width = 2.5
    
    fig_width = 2*header_width + max_cols * cell_width + 0.5
    fig_height = (num_rows + 1) * cell_height + 0.5
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.set_xlim(0, 2*header_width + max_cols * cell_width)
    ax.set_ylim(0, (num_rows + 1) * cell_height)
    ax.axis('off')
    
    # Draw init header cell
    header_rect = mpatches.Rectangle(
        (0, num_rows * cell_height), header_width, cell_height,
        edgecolor='white', facecolor='#5a5a5a', linewidth=2
    )
    ax.add_patch(header_rect)
    ax.text(3*header_width/2, (num_rows + 0.5) * cell_height, title,
            ha='center', va='center', color='white', fontsize=12, weight='bold')
    
    # Draw name header cell
    header_rect = mpatches.Rectangle(
        (header_width, num_rows * cell_height), header_width, cell_height,
        edgecolor='white', facecolor='#5a5a5a', linewidth=2
    )
    ax.add_patch(header_rect)
    ax.text(header_width/2, (num_rows + 0.5) * cell_height, "Name",
            ha='center', va='center', color='white', fontsize=12, weight='bold')
    
    # Draw each row
    if nameMap != None:
        data_dict = {nameMap.get(k, k): v for k, v in data_dict.items()}
    for row_idx, (label, values) in enumerate(data_dict.items()):
        y_pos = (num_rows - row_idx - 1) * cell_height
        
        # Draw name value cell (white background)
        init_rect = mpatches.Rectangle(
            (0, y_pos), header_width, cell_height,
            edgecolor='#cccccc', facecolor='white', linewidth=1
        )
        ax.add_patch(init_rect)
        
        # Add name value 
        ax.text(header_width/2, y_pos + cell_height/2, str(label),
                ha='center', va='center', color='black', fontsize=11)
        
        # Draw initial value cell (white background) - shows first actual value
        init_rect = mpatches.Rectangle(
            (header_width, y_pos), header_width, cell_height,
            edgecolor='#cccccc', facecolor='white', linewidth=1
        )
        ax.add_patch(init_rect)
        
        # Add initial value (first value from the data)
        ax.text(3*header_width/2, y_pos + cell_height/2, str(values[0]),
                ha='center', va='center', color='black', fontsize=11)
        
        # Draw remaining value cells
        for col_idx in range(len(values)):
            value = values[col_idx]
            #x_pos = header_width + col_idx * cell_width
            
            # For the first column, we already drew it in the header section
            if col_idx == 0:
                continue
            
            # Get previous value
            prev_value = values[col_idx - 1]
            
            # Determine color
            if col_idx in repeated_cols:
                # Gray for repeated state
                color = '#d4d4d4'
            elif value != prev_value:
                # Yellow if value changed
                color = '#f5f5c8'
            else:
                # White if value same as previous
                color = '#ffffff'
            
            # Draw cell (offset by -1 since we skip the first column)
            cell_rect = mpatches.Rectangle(
                (2*header_width + (col_idx - 1) * cell_width, y_pos), cell_width, cell_height,
                edgecolor='#cccccc', facecolor=color, linewidth=1
            )
            ax.add_patch(cell_rect)
            
            # Add value text
            ax.text(2*header_width + (col_idx - 1) * cell_width + cell_width/2, 
                   y_pos + cell_height/2, str(value),
                   ha='center', va='center', color='black', fontsize=11)
    
    plt.tight_layout()
    plt.show()

In [6]:
def generate_gene_combinations(num_genes, num_values):
    """
    Generates combinations where the first gene is 
    smaller than all subsequent genes.
    """
    # Possible values for each gene
    values = range(0, num_values)
    
    # Generate all possible Cartesian products for the given number of genes
    # Example: (0,0,0), (0,0,1) ... (3,3,3)
    all_combinations = itertools.product(values, repeat=num_genes)
    
    valid_combinations = []
    if num_genes > 1 :
        for combo in all_combinations:
            first_gene = combo[0]
            other_genes = combo[1:]
            
            # Rule: First element must be lower than ALL others
            # We use 'all()' to ensure the condition met for every subsequent gene
            if all(first_gene < other for other in other_genes):
                valid_combinations.append(combo)
    else:
        valid_combinations.append((0,))
        valid_combinations.append((1,))
        valid_combinations.append((2,))
        valid_combinations.append((3,))
            
    return valid_combinations

In [7]:
def B_cell_simulation(model, combinaison, timing, output_dir, prefix = 'BCRABL', treatment = None, nameMap = None, ):
    """
    Export simulation results to CSV using combination parameters for naming.

    This function iterates through gene combinations, constructs a filename
    based on the gene, timing stage, and mutation status, and writes the
    corresponding simulation data to a CSV file.

    Args:
        combinaison: Dictionary containing gene groupings and statuses.
        timing: Dictionary mapping time indices to stage names (e.g., {0: 'CLP'}).
        output_dir: The directory where the CSV files will be saved.
        nameMap: Dictionary mapping node ID to thier name
        prefix: The leading string for the filename.
    """
    # Ensure a directory exists for the outputs
 # Initialize storage and directory
    simulations = []
    all_combinaisons = []
    genes_lists = []

    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for primary_gene, groups in combinaison.items():
        for entry in groups:
            
            sub_genes = list(entry.keys())
            
            # 1. Generate combinations (Ensure single genes are tuples)
            if len(sub_genes) > 1:
                raw_combinations = generate_gene_combinations(len(sub_genes), 4)
            else:
                # Standardizes format to list of tuples: [(0,), (1,), (2,), (3,)]
                raw_combinations = [(0,), (1,), (2,), (3,)]
                
            # 2. Map genes to their timing values
            genes_list = [dict(zip(sub_genes, combo)) for combo in raw_combinations]
            
            # Track for debugging/history
            genes_lists.append(genes_list)
            all_combinaisons.append(raw_combinations)
    
            # 3. Run Simulation Loop
            for t in range(len(genes_list)):
                current_combo_dict = genes_list[t] # The dict: {'Ikzf1': 0}
                current_combo_tuple = raw_combinations[t] # The tuple: (0,)
                
                mod = copy.deepcopy(model)    
                var_lookup = {var["Name"]: var for var in mod["Model"]["Variables"]}
                var_lookup['BCR-ABL']['RangeTo'] = 2
                
                for gene_name, status in entry.items():
                    if gene_name in var_lookup:
                        target_var = var_lookup[gene_name]
                        
                        # Get timing value and handle tuple vs int
                        val = current_combo_dict[gene_name]
                        timing_index = val[0] if isinstance(val, tuple) else val
                        
                        if gene_name in ('Ikzf1', 'Pax5'):
                            node_to_update = Formulas[gene_name]['Node']
                            target_formula_list = Formulas[gene_name]['Formula']
                            var_lookup[node_to_update]['Formula'] = target_formula_list[timing_index]
                            target_var['RangeTo'] = Formulas[gene_name][status]
                             
                        else:
                            node_to_update = Formulas[gene_name]['Node']
                            target_formula_list = Formulas[gene_name]['Formula']
                            var_lookup[node_to_update]['Formula'] = target_formula_list[timing_index]  

                    qn2 = pybma.model_to_qn(mod)
                    sim_result = pybma.simulate(qn2, steps=30)
                    simulations.append(sim_result)
        
                # 4. Filename Generation
                name_parts = [prefix]
                for i, gene in enumerate(sub_genes):
                    # Access the tuple index safely
                    t_idx = current_combo_tuple[i]
                    time_stage = timing.get(t_idx, "Unknown")
                    mut_status = entry[gene].upper()
                    name_parts.extend([gene.upper(), time_stage, mut_status])
                
                filename = "-".join(name_parts) + ".csv"
                file_path = os.path.join(output_dir, filename)
    
                # 5. Write to CSV (Row format: [GeneID, val1, val2...])
                with open(file_path, mode='w', newline='') as f:
                    writer = csv.writer(f)
                    
                    # Apply nameMap if it exists
                    if nameMap is not None:
                        data_dict = {nameMap.get(k, k): v for k, v in sim_result.items()}
                    else:
                        data_dict = sim_result
                    
                    for key, values in data_dict.items():
                        writer.writerow(['',key] + values)

In [20]:
all_combinaisons = []

mod = copy.deepcopy(m)    
var_lookup = {var["Name"]: var for var in mod["Model"]["Variables"]}
var_lookup['BCR-ABL']['RangeTo'] = 2

mod_mut = pybma.model_to_qn(mod)
sim_result = pybma.simulate(mod_mut, steps=30)
variables = list(sim_result.keys())
state = {}

genes_lists = []

for primary_gene, groups in combinaison.items():
    for entry in groups:
        sub_genes = list(entry.keys())
        
        # 1. Generate combinations (Ensure single genes are tuples)
        if len(sub_genes) > 1:
            raw_combinations = generate_gene_combinations(len(sub_genes), 28)
        else:
            # Standardizes format to list of tuples: [(0,), (1,), (2,), (3,)]
            raw_combinations = []
            for i in range(28):
                raw_combinations.append((i,))
            
        # 2. Map genes to their timing values
        genes_list = [
            dict(sorted(zip(sub_genes, combo), key=lambda item: item[1]))
            for combo in raw_combinations
        ]        
                # Track for debugging/history
        genes_lists.append(genes_list)
        all_combinaisons.append(raw_combinations)
         # 3. Run Simulation Loop
        for t in range(len(genes_list)):
            simulations = []
            
            current_combo_dict = genes_list[t]
            current_combo_tuple = sorted(raw_combinations[t]) # The tuple: (0,)
            # current_combo_dict = dict(sorted(current_combo_dict.items(), key=lambda item: item[1]))

            
            print(current_combo_dict)
           # update the values for the second, third etc genes as the steps wanted is based on the BCR-ABL simulation
            processed_data = {}
            previous_value = 0
            for key, current_value in current_combo_dict.items():
                # If values are the same, no subtraction occurs
                if current_value == previous_value or current_value == same_value:
                    new_value = 0
                    same_value = current_value
                else:
                    new_value = current_value - previous_value
                    same_value = current_value
                
                processed_data[key] = new_value
                
                # Update previous_value for the next iteration
                previous_value = current_value

            current_combo_dict = processed_data

            processed_list = []
            previous_value = 0
            
            for current_value in current_combo_tuple:
    # Apply your logic: no subtraction if values are identical
                if current_value == previous_value or current_value == same_value:
                    new_value = 0
                    same_value = current_value
                else:
                    new_value = current_value - previous_value
                    same_value = current_value
                
                processed_list.append(new_value)
                previous_value = current_value
            
            # Convert the list back to a tuple
            current_combo_tuple = tuple(processed_list)

            
            print(current_combo_dict)
            
            state = {}
            for gene_name in entry:
                for var in variables:
                    state[var] = sim_result[var][current_combo_dict[gene_name]]
                    
            #extract the n first step of the BCR-ABL only simulation
            n = current_combo_tuple[0]
            # Slice each list up to index n
            trimmed_data = {key: values[:n] for key, values in sim_result.items()}
            simulations.append(trimmed_data)
                        
            for i, (gene_name, status) in enumerate(entry.items()):                
                if gene_name in var_lookup:
                    target_var = var_lookup[gene_name]
                    
                    # Get timing value and handle tuple vs int
                    var_lookup[gene_name]['Formula'] = Formulas_steps[gene_name][status]
                    
                    if i+1 < len(entry): 
                        num_step= max(3,current_combo_tuple[i+1]+1)
                        
                        print('steps '+str(num_step))
                        print('Gene step '+str(current_combo_tuple[i]))

                        mod_mut = pybma.model_to_qn(mod)
                        mutant_simulation = pybma.simulate(mod_mut,steps=num_step,initial_values=state)
                        
                        print('Sim ' +str(len(mutant_simulation[74])))
                        print('Gene step '+str(current_combo_dict[gene_name]))
                        
                        print(mutant_simulation)
                        n = current_combo_tuple[i+1]
                        trimmed_data = {key: values[:n] for key, values in mutant_simulation.items()}
                        simulations.append(trimmed_data)
                        for var in variables:
                            state[var] = mutant_simulation[var][current_combo_tuple[i+1]]
                    else :
                        mod_mut = pybma.model_to_qn(mod)
                        mutant_simulation = pybma.simulate(mod_mut,steps=30,initial_values=state)
                        for var in variables:
                            state[var] = mutant_simulation[var][current_combo_tuple[i]]
                            
                        print('Sim ' +str(len(mutant_simulation[74])))
                        print('Gene step '+str(current_combo_dict[gene_name]))

                    
                    
            mod_mut = pybma.model_to_qn(mod)
            final_step = 30 - max(current_combo_tuple)
            simulations.append(pybma.simulate(mod_mut,steps=final_step,initial_values=state))
            
            all_keys = simulations[0].keys()

            # Fuse the dictionaries and flatten the lists
            fused_combinaison = {
                key: [val for d in simulations for val in d.get(key, [])]
                for key in all_keys
            }

            name_parts = [prefix]
            for i, gene in enumerate(sub_genes):
                # Access the tuple index safely
                t_idx = current_combo_tuple[i]
                time_stage = 'steps_' + str(t_idx)
                mut_status = entry[gene].upper()
                name_parts.extend([gene.upper(), time_stage, mut_status])
            
            filename = "-".join(name_parts) + ".csv"
            file_path = os.path.join(output_dir, filename)

            # 5. Write to CSV (Row format: [GeneID, val1, val2...])
            with open(file_path, mode='w', newline='') as f:
                writer = csv.writer(f)
                
                # Apply nameMap if it exists
                if nameMap is not None:
                    data_dict = {nameMap.get(k, k): v for k, v in fused_combinaison.items()}
                else:
                    data_dict = fused_combinaison
                
                for key, values in data_dict.items():
                    writer.writerow(['',key] + values)

            

            

{'EBF1': 0, 'Ikzf1': 1, 'Pax5': 1, 'CDKN2A': 1}
{'EBF1': 0, 'Ikzf1': 1, 'Pax5': 0, 'CDKN2A': 0}
steps 3
Gene step 0
Sim 2
Gene step 0
{2: [0, 0], 3: [0, 1], 7: [0, 0], 8: [0, 0], 11: [0, 0], 17: [0, 1], 19: [0, 0], 30: [0, 0], 38: [0, 0], 59: [0, 1], 68: [0, 0], 74: [0, 0], 120: [0, 0], 121: [0, 0], 122: [0, 0], 126: [0, 0], 130: [0, 0], 133: [0, 0], 146: [0, 1], 147: [0, 0], 148: [0, 0], 149: [0, 0], 164: [0, 0], 171: [0, 0], 179: [0, 0], 180: [0, 0], 181: [0, 0], 182: [0, 0], 198: [0, 0], 206: [0, 0], 211: [0, 0], 231: [0, 1], 232: [0, 0], 236: [0, 0], 237: [0, 0], 247: [0, 0], 254: [0, 0], 257: [0, 0], 266: [0, 0], 272: [0, 1], 275: [0, 1], 280: [0, 1], 293: [0, 0], 307: [0, 0], 313: [0, 0], 322: [0, 0], 331: [0, 0], 343: [0, 0], 344: [0, 0], 366: [0, 0], 383: [0, 0], 390: [0, 0], 392: [0, 0], 408: [0, 0], 414: [0, 0], 415: [0, 0], 423: [0, 0], 424: [0, 0], 425: [0, 0], 437: [0, 0], 457: [0, 0], 464: [0, 0], 471: [0, 0], 472: [0, 0], 473: [0, 0]}
steps 3
Gene step 1
Sim 2
Gene step 

IndexError: list index out of range

In [18]:
m = pybma.load_model("B-diff-pyBMA.json")
# pybma.save_model(m,"B-diff-pyBMA.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)

nameMap = pybma.model_to_variableIDdict(m)

prefix='BCRABL'
output_dir = './simulations/steps/'

In [193]:
var_lookup = {var["Name"]: var for var in m["Model"]["Variables"]}
print(var_lookup['Foxo1'])


{'Name': 'Foxo1', 'Id': 8, 'RangeFrom': 0, 'RangeTo': 2, 'Formula': 'floor(avg(var(74),(2*var(7)-2),var(17)))-var(307)'}


In [12]:
combinaison = {
     'EBF1': [
         # {'EBF1': 'HETERO'},
         # {'EBF1': 'HETERO', 'CDKN2A': 'HOMO'},
         # {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'},
         # {'EBF1': 'HETERO', 'Ikzf1': 'HOMO'}, 
         # {'EBF1': 'HETERO', 'Ikzf1': 'HETERO'}, 
         {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
         # {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
         # {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
         # {'EBF1': 'HETERO', 'Pax5': 'HETERO'},
         # {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO'}
      ],
    #  'Ikzf1': [
    #      {'Ikzf1': 'HOMO'},
    #      {'Ikzf1': 'HETERO'},
    #      {'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
    #      {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO'},
    #      {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'}
    #  ],
    # 'Pax5': [
    #     {'Pax5': 'HETERO'},
    #     {'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
    #     {'Pax5': 'HETERO', 'Ikzf1': 'HETERO'},
    #     {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'}        
    # ]
}                                

# Parsing logic

Formulas = { 
    'EBF1' : {'Node' : 'EBF1_KO', 'Formula' : ['var(146)+var(457)', 'var(147)+var(457)', 'var(148)+var(457)', 'var(149)+var(457)']},
    'Pax5': {'Formula' : ['var(146)+var(464)', 'var(147)+var(464)', 'var(148)+var(464)', 'var(149)+var(464)'],
                            'HOMO' : 0,'HETERO' : 2, 'Node' : 'PAX5_KO'},
    'Ikzf1' : {'Formula' : ['var(146)+var(408)', 'var(147)+var(408)', 'var(148)+var(408)', 'var(149)+var(408)'],
                            'HOMO' : 0,'HETERO' : 2, 'Node' : 'IKZF1_ko'},
    'CDKN2A' : {'Node' : 'KO_CDKN2A', 'Formula' : ['var(146)+var(425)', 'var(147)+var(425)', 'var(148)+var(425)', 'var(149)+var(425)']},
    'BCR-ABL' : {'Node' : 'KO_BCRABL', 'Formula' : ['var(146)+var(473)','var(147)+var(473)', 'var(148)+var(473)', 'var(149)+var(473)']},
    'JAK-STAT' : {'Node' : 'KO_JAK_STAT', 'Formula' : ['var(146)+var(392)', 'var(147)+var(392)', 'var(148)+var(392)', 'var(149)+var(392)']},
    'MAPK' : {'Node' : 'KO_MAPK', 'Formula' : ['var(146)+var(472)', 'var(147)+var(472)', 'var(148)+var(472)', 'var(149)+var(472)']},
    'PIP3' : {'Node' : 'KO_PIP3', 'Formula' : ['var(146)+var(471)', 'var(147)+var(471)', 'var(148)+var(471)', 'var(149)+var(471)']}
}

Formulas_steps = { 
    'EBF1' : {'HETERO' : 'min(1,floor(((var(8)+var(59)+var(11)+var(171)+var(74))/5)+1/3)+(2*var(7)-2)-max(0,(2*var(206)-var(211)))-var(237)/2)'},
    'Pax5': {'HOMO' : '0','HETERO' : 'min(1, floor(avg(var(74),var(8)))+max(0,(2*var(11)-2))-var(275)/2 +2*var(437))'},
    'Ikzf1' : {'HOMO' : '0','HETERO' : 'min(1, 2+var(8))'},
    'CDKN2A' :{ 'HOMO' : '0'}  
}
timing = {
    0 : 'CLP',
    1 : 'PreproB',
    2 : 'ProB',
    3 : 'PreB'
}

In [180]:
prefix='BCRABL'
output_dir = './simulations/steps/'
# B_cell_simulation(m, combinaison, timing, output_dir, prefix, nameMap = nameMap)

In [33]:
combinaison = {
    'EBF1': [
        # Base: {'EBF1': 'HETERO'} + 16 combos of BCR-ABL, PIP3, MPAK, JAK-STAT
        {'EBF1': 'HETERO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Ikzf1': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Ikzf1': 'HETERO'} + 16 combos
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO'} + 16 combos
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Pax5': 'HETERO'} + 16 combos
        {'EBF1': 'HETERO', 'Pax5': 'HETERO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO'} + 16 combos
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'}
    ],
    'Ikzf1': [
        # Base: {'Ikzf1': 'HOMO'} + 16 combos
        {'Ikzf1': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'Ikzf1': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Ikzf1': 'HETERO'} + 16 combos
        {'Ikzf1': 'HETERO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'Ikzf1': 'HETERO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HETERO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Ikzf1': 'HOMO', 'Pax5': 'HETERO'} + 16 combos
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO'} + 16 combos
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'}
    ],
    'Pax5': [
        # Base: {'Pax5': 'HETERO'} + 16 combos
        {'Pax5': 'HETERO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'Pax5': 'HETERO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Pax5': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Pax5': 'HETERO', 'Ikzf1': 'HETERO'} + 16 combos
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},

        # Base: {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'} + 16 combos
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'},
        {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO', 'BCR-ABL': 'HOMO', 'PIP3': 'HOMO', 'MPAK': 'HOMO', 'JAK-STAT': 'HOMO'}
    ]
}